# TPL Spacing Calculations

## 1. Importing / Installing Packages

In [1]:
import os

import pandas as pd
pd.set_option('display.max_columns', None)

from src.utils import DatabricksConfig, get_logger, compute_bg_rcat, reorder_columns
from src.well_data import WellDataLoader, GeoSurveyProcessor

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'svg' # Configuring inline backend to use SVG format for figures

## 2. Initializing Logger

In [2]:
logger = get_logger(name="spacing_tpl", log_to_console=False, level="INFO", propagate=True)

## 3. Importing Header and Directional Survey

### 3.1 Initializing Database Configuration 

In [3]:
cfg = DatabricksConfig(
server_hostname="adb-3250236208859616.16.azuredatabricks.net",
http_path="/sql/1.0/warehouses/0cf5fe590cb7b313",
token=os.environ["databricks_token"],
catalog="bronze",
schema="enverus",
)

loader = WellDataLoader(db_cfg=cfg,
                        logger=logger, directional_source="enverus")

### 3.2 Importing Header

In [4]:
header_query = """
    SELECT
        API_UWI_14_Unformatted AS uwi,
        API_UWI_12_Unformatted AS uwi12,
        LeaseName AS lease_name,
        WellName AS well_name,
        ENVOperator AS operator,
        ENVInterval AS bench,
        SpudDate as spud_date,
        CompletionDate AS comp_date,
        FirstProdDate AS first_prod_date,
        LastProducingMonth as last_prod_date,
        Trajectory AS hole_direction,
        ENVWellStatus as well_status,
        Latitude AS surface_lat,
        Longitude AS surface_lon
    FROM bronze.enverus.wells
    WHERE 
        StateProvince = 'TX'
            AND
        County IN ('MIDLAND', 'HOWARD', 'MARTIN', 'GLASSCOCK')
            AND
        Trajectory = 'HORIZONTAL'
"""

df_header_db = loader.get_header_data(header_query=header_query)

[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.


### 3.3 Computing BG Reserve Category

In [5]:
df_header_db['bg_rcat'] = compute_bg_rcat(df_header_db, 
                col_map={
    "status": "well_status",
    "last_prod": "last_prod_date",
    "spud": "spud_date",
    "comp": "comp_date"
                        },
    utc=True
)

df_header_db = reorder_columns(df_header_db,columns_to_move=['bg_rcat'],reference_column='bench')

In [11]:
df_header_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22998 entries, 0 to 22997
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   uwi              22998 non-null  object             
 1   uwi12            22998 non-null  object             
 2   lease_name       22936 non-null  object             
 3   well_name        22936 non-null  object             
 4   operator         22936 non-null  object             
 5   bench            22773 non-null  object             
 6   bg_rcat          22998 non-null  object             
 7   spud_date        20304 non-null  datetime64[ns, UTC]
 8   comp_date        18181 non-null  datetime64[ns, UTC]
 9   first_prod_date  18513 non-null  datetime64[ns, UTC]
 10  last_prod_date   17454 non-null  datetime64[ns, UTC]
 11  hole_direction   22998 non-null  object             
 12  well_status      22998 non-null  object             
 13  surface_lat     

In [6]:
df_header_db

,uwi,uwi12,lease_name,well_name,operator,bench,bg_rcat,spud_date,comp_date,first_prod_date,last_prod_date,hole_direction,well_status,surface_lat,surface_lon
0,42329476420000,423294764200,RAY-SWEENEY 43N,RAY-SWEENEY 43N 14H,EXXON,WOLFCAMP A,2DUC,2025-05-27 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.907771,-101.895878
1,42329475330000,423294753300,TRIPLE HOP,TRIPLE HOP 25JM,OCCIDENTAL,JO MILL,1WOP,2025-09-15 00:00:00+00:00,2025-12-20 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.914081,-101.866252
2,42173392890000,421733928900,DRIVER B37 10-15K,DRIVER B37 10-15K 311H,EXXON,WOLFCAMP A,1WOP,2025-06-13 00:00:00+00:00,2025-11-16 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.681581,-101.770590
3,42329477300000,423294773000,DRIVER B37 10-15C,DRIVER B37 10-15C 303H,EXXON,WOLFCAMP A,1WOP,2025-06-08 00:00:00+00:00,2025-10-31 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.680906,-101.781241
4,42173393200000,421733932000,SCHWARTZ Q,SCHWARTZ Q 7A,CRESCENT ENERGY COMPANY,WOLFCAMP A,2DUC,2025-07-31 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.782222,-101.558022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22993,42329479360000,423294793600,MARSH 36G,MARSH 36G 107H,EXXON,WOLFCAMP D,2DUC,2026-01-20 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DRILLING,31.748589,-102.191065
22994,42329392620000,423293926200,OLDHAM,OLDHAM 1H,ROCK FISH OPERATING LLC,None,1PDP,2014-06-08 00:00:00+00:00,2014-07-23 00:00:00+00:00,2014-08-01 00:00:00+00:00,2025-06-01 00:00:00+00:00,HORIZONTAL,PRODUCING,32.070790,-101.908217
22995,42173392270000,421733922700,FLINTLOCK E,FLINTLOCK E 11JM,OCCIDENTAL,JO MILL,2DUC,2025-11-26 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.831036,-101.703714
22996,42317471340000,423174713400,CURTIS-SCHARBAUER 8C,CURTIS-SCHARBAUER 8C 3H,EXXON,WOLFCAMP C,2DUC,2026-01-19 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,SPUD DATE ONLY,32.246788,-102.124859


In [7]:
df_header_db['bg_rcat'].value_counts()

bg_rcat
1PDP     17082
9XPMT     1979
2DUC      1090
9XDUC     1026
1WOP       612
3PRMT      583
1PDSI      247
           238
9PA        141
Name: count, dtype: int64

In [8]:
df_header_db[df_header_db['bg_rcat']==""]

,uwi,uwi12,lease_name,well_name,operator,bench,bg_rcat,spud_date,comp_date,first_prod_date,last_prod_date,hole_direction,well_status,surface_lat,surface_lon
16,42173334510200,421733345102,BEARKAT,BEARKAT 1501H,CRESCENT ENERGY COMPANY,EASTERN SHELF ALL,,2009-03-07 00:00:00+00:00,NaT,2009-06-01 00:00:00+00:00,NaT,HORIZONTAL,INACTIVE PRODUCER,31.854127,-101.409309
54,42317345350200,423173453502,DAYTONA 66,DAYTONA 66 1HW,EXXON,WOODFORD AND BELOW,,2005-05-02 00:00:00+00:00,NaT,2005-08-01 00:00:00+00:00,NaT,HORIZONTAL,INACTIVE INJECTOR,32.415230,-101.790530
97,42329481240000,423294812400,GBG PERSEUS 35-14 A,GBG PERSEUS 35-14 A 2152US,CHEVRON,LOWER PENNSYLVANIAN AND MISSISSIPPIAN,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.891644,-102.151428
98,42329401210001,423294012100,DORA ROBERTS RANCH UNIT,DORA ROBERTS RANCH UNIT 4038H,CONTINENTAL RESOURCES,LOWER SPRABERRY,,2015-03-13 00:00:00+00:00,2017-04-13 00:00:00+00:00,2015-06-01 00:00:00+00:00,NaT,HORIZONTAL,INACTIVE PRODUCER,31.735134,-102.234359
113,42329481260000,423294812600,GBG PERSEUS 35-14 C,GBG PERSEUS 35-14 C 2156US,CHEVRON,LOWER PENNSYLVANIAN AND MISSISSIPPIAN,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.891703,-102.151180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22833,42329481060000,423294810600,None,None,None,None,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.828184,-101.984234
22852,42329481150000,423294811500,None,None,None,None,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.976112,-101.994309
22861,42329481160000,423294811600,None,None,None,None,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.976126,-101.994247
22863,42329481030000,423294810300,None,None,None,None,,NaT,NaT,NaT,NaT,HORIZONTAL,UNREPORTED,31.828132,-101.984467


### 3.4 Filtering Header to include PDP, WOP, PDSI and DUC Rsv Categories

In [9]:
df_header_rsv_filter = df_header_db[df_header_db['bg_rcat'].isin(["1PDP", "1WOP", "1PDSI", "2DUC"])].reset_index(drop=True).copy()

In [10]:
df_header_rsv_filter

,uwi,uwi12,lease_name,well_name,operator,bench,bg_rcat,spud_date,comp_date,first_prod_date,last_prod_date,hole_direction,well_status,surface_lat,surface_lon
0,42329476420000,423294764200,RAY-SWEENEY 43N,RAY-SWEENEY 43N 14H,EXXON,WOLFCAMP A,2DUC,2025-05-27 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.907771,-101.895878
1,42329475330000,423294753300,TRIPLE HOP,TRIPLE HOP 25JM,OCCIDENTAL,JO MILL,1WOP,2025-09-15 00:00:00+00:00,2025-12-20 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.914081,-101.866252
2,42173392890000,421733928900,DRIVER B37 10-15K,DRIVER B37 10-15K 311H,EXXON,WOLFCAMP A,1WOP,2025-06-13 00:00:00+00:00,2025-11-16 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.681581,-101.770590
3,42329477300000,423294773000,DRIVER B37 10-15C,DRIVER B37 10-15C 303H,EXXON,WOLFCAMP A,1WOP,2025-06-08 00:00:00+00:00,2025-10-31 00:00:00+00:00,NaT,NaT,HORIZONTAL,COMPLETED,31.680906,-101.781241
4,42173393200000,421733932000,SCHWARTZ Q,SCHWARTZ Q 7A,CRESCENT ENERGY COMPANY,WOLFCAMP A,2DUC,2025-07-31 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.782222,-101.558022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19026,42329479360000,423294793600,MARSH 36G,MARSH 36G 107H,EXXON,WOLFCAMP D,2DUC,2026-01-20 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DRILLING,31.748589,-102.191065
19027,42329392620000,423293926200,OLDHAM,OLDHAM 1H,ROCK FISH OPERATING LLC,None,1PDP,2014-06-08 00:00:00+00:00,2014-07-23 00:00:00+00:00,2014-08-01 00:00:00+00:00,2025-06-01 00:00:00+00:00,HORIZONTAL,PRODUCING,32.070790,-101.908217
19028,42173392270000,421733922700,FLINTLOCK E,FLINTLOCK E 11JM,OCCIDENTAL,JO MILL,2DUC,2025-11-26 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,DUC,31.831036,-101.703714
19029,42317471340000,423174713400,CURTIS-SCHARBAUER 8C,CURTIS-SCHARBAUER 8C 3H,EXXON,WOLFCAMP C,2DUC,2026-01-19 00:00:00+00:00,NaT,NaT,NaT,HORIZONTAL,SPUD DATE ONLY,32.246788,-102.124859


### 3.5 Filtering Header to include PDP, WOP, PDSI and DUC Rsv Categories

In [12]:
uwis_12_list = df_header_rsv_filter['uwi12'].astype(str).dropna().unique().tolist()

In [13]:
directional_query = """
    SELECT
        API_UWI_12_Unformatted AS uwi12,
        MeasuredDepth_FT AS md,
        TVD_FT AS tvd,
        Inclination_DEG AS inclination,
        Azimuth_DEG AS azimuth,
        Latitude AS latitude,
        Longitude AS longitude,
        E_W AS `deviation_E/W`,
        N_S AS `deviation_N/S`
    FROM bronze.enverus.fulldirectionalsurvey
    WHERE 
        API_UWI_12_Unformatted IN :uwis_12
    ORDER BY 
        API_UWI_12_Unformatted, MeasuredDepth_FT;
"""

df_directional_db = loader.get_directional_data(directional_query=directional_query, 
                                                directional_params={"uwis_12": uwis_12_list})

[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.
[WARN] Parame

In [16]:
missing_uwi = set(df_header_rsv_filter['uwi12']) - set(df_directional_db['uwi12'])
print(f"Number of missing UWI: {len(missing_uwi)}")
# list(missing_uwi)

Number of missing UWI: 17


In [27]:
df_directional_db_merge_uwi = df_directional_db.merge(df_header_rsv_filter[['uwi','uwi12']].drop_duplicates(subset=['uwi12'], keep='first'), on='uwi12', how='left')

df_directional_db_merge_uwi = reorder_columns(df_directional_db_merge_uwi,columns_to_move=['uwi'],reference_column='uwi12')

## 3. Computing UTM Coordinates

In [32]:
geo = GeoSurveyProcessor(logger=logger) # Initialize GeoSurveyProcessor

df_utm = geo.compute_utm_coordinates(df=df_directional_db_merge_uwi)

# Filter the DataFrame to get only the lateral sections after the heel point
df_utm_lateral = geo.filter_after_heel_point(df=df_utm)

# Calculate midpoints for lateral wells
df_midpoints_lateral = geo.get_heel_toe_midpoints_latlon(df=df_utm_lateral)